# Metadata Aggregator Notebook

## System set up

In [1]:
%%capture
%pip install pydash

In [2]:
import json
import os
import uuid

import numpy as np
import pandas as pd

from pydash import py_

## Load data

In [3]:
data_path = ".."

in_colab = "google.colab" in str(get_ipython())

if in_colab:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    data_path = os.path.join("drive", "MyDrive", "Crossreads B D1", "crossreads_petrography_data")

!ls "{data_path}"

Isotopes            Metadata            Tools               default_config.yaml
MGS                 README.md           XRD                 pXRF


### Metamorphic

In [4]:
mm_path = os.path.join(data_path, 'Metadata/Metamorphic.xlsx')
mm_df = pd.read_excel(mm_path)
mm_df.columns = mm_df.columns.str.strip()

mm_row_0 = mm_df.loc[0].copy()
mm_row_0['Inscription'] = 'from XML'

mm_df.head()

,ISic,Ancient place,Date after,Date before,Type of text,Execution,Language,Link to image,Museum,ref,...,Gd 158_2_1_1,Dy 164_2_1_1,Er 166_2_1_1,Yb 174_2_1_1,Hf 180_2_1_1,W 184_2_1_1,Pb 208_2_1_1,Bi 209_2_1_1,Th 232_2_1_1,U 238_2_1_1
0,AAA,from XML,from XML,from XML,from XML,from XML,from XML,from XML,from XML,REF,...,MANUAL INPUT,MANUAL INPUT,MANUAL INPUT,MANUAL INPUT,MANUAL INPUT,MANUAL INPUT,MANUAL INPUT,MANUAL INPUT,MANUAL INPUT,MANUAL INPUT
1,CU431,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CU477,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CU478,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,EXMFT003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Sedimentary

In [5]:
sd_path = os.path.join(data_path, 'Metadata/Sedimentary.xlsx')
sd_df = pd.read_excel(sd_path)
sd_df.columns = sd_df.columns.str.strip()

sd_row_0 = sd_df.loc[0].copy()
sd_row_0['Inscription'] = 'from XML'

sd_df.head()

,ISic,Ancient place,Date after,Date before,Type of text,Execution,Language,Link to image,Museum,ref,...,Mn,Nb,Pb,Rb,Sr,Th,Ti,Y,Zn,Zr
0,AAA,from XML,from XML,from XML,from XML,from XML,from XML,from XML,from XML,REF,...,XRF TOOL (header in pXRF_calculated_fractions_...,XRF TOOL (header in pXRF_calculated_fractions_...,XRF TOOL (header in pXRF_calculated_fractions_...,XRF TOOL (header in pXRF_calculated_fractions_...,XRF TOOL (header in pXRF_calculated_fractions_...,XRF TOOL (header in pXRF_calculated_fractions_...,XRF TOOL (header in pXRF_calculated_fractions_...,XRF TOOL (header in pXRF_calculated_fractions_...,XRF TOOL (header in pXRF_calculated_fractions_...,XRF TOOL (header in pXRF_calculated_fractions_...
1,HAL18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ISic000007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ISic000009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ISic000009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Aggregate data

In [6]:
def get_isic_rows(df):
    """Returns only the rows that are present in I.Sicily."""
    return df[df['ISic'].str.startswith('ISic')]

### Epidoc data

In [7]:
inscriptions_path = os.path.join(data_path, 'Metadata/corpus.json')
with open(inscriptions_path, 'r') as f:
    inscriptions_list = json.load(f)

inscriptions = {i["file"]: i for i in inscriptions_list}

print(len(inscriptions.keys()))

4735


In [8]:
def get_date(x, field):
    if x[f'Date {field.lower()}']:
        return x[f'Date {field.lower()}']

    return py_.get(inscriptions, f"{x['ISic']}.date.not{field}")

In [9]:
def get_inscription_type(isic):
    return py_.get(inscriptions, f'{isic}.type._')

In [10]:
def get_execution(isic):
    rs = py_.get(inscriptions, f'{isic}.layoutDesc.layout.rs')

    if rs is None:
        return None

    ana = None

    if isinstance(rs, list):
        ana = py_.get(rs, '0.ana')
    else:
        ana = rs.get('ana')

    if ana is None:
        return None

    return ', '.join(ana.split('.')[1:])

In [11]:
def get_language(isic):
    languages = py_.get(inscriptions, f'{isic}.textLang.languages')

    if languages is None:
        return None

    return ', '.join(languages)

In [12]:
image_server = 'https://apheleia.classics.ox.ac.uk/iipsrv/iipsrv.fcgi?IIIF=inscription_images/'
image_params = '/full/full/0/default.jpg'

def get_image_url(isic):
    url = py_.get(inscriptions, f'{isic}.facsimile.url')

    if url is None:
        return None

    return f'=HYPERLINK("{image_server}{isic}/{url}{image_params}", "{url}")'

In [13]:
def get_repository(isic):
    return py_.get(inscriptions, f'{isic}.repository._')

In [14]:
for df in [mm_df, sd_df]:
    df['ISic'] = df['ISic'].fillna(uuid.uuid4()).astype(str)

    df['Ancient place'] = df['ISic'].apply(lambda x: py_.get(inscriptions, f'{x}.places.0._'))

    df['Date after'] = df.apply(lambda x: get_date(x, 'After'), axis=1)
    df['Date before'] = df.apply(lambda x: get_date(x, 'Before'), axis=1)

    df['Type of text'] = df['ISic'].apply(get_inscription_type)
    df['Execution'] = df['ISic'].apply(get_execution)
    df['Language'] = df['ISic'].apply(get_language)
    df['Link to image'] = df['ISic'].apply(get_image_url)
    df['Museum'] = df['ISic'].apply(get_repository)
    df['Inscription'] = df['ISic'].apply(
        lambda x: f'=HYPERLINK("http://sicily.classics.ox.ac.uk/inscription/{x}", "{x}")'
        if x in inscriptions else None
    )

In [15]:
get_isic_rows(mm_df).head()

,ISic,Ancient place,Date after,Date before,Type of text,Execution,Language,Link to image,Museum,ref,...,Dy 164_2_1_1,Er 166_2_1_1,Yb 174_2_1_1,Hf 180_2_1_1,W 184_2_1_1,Pb 208_2_1_1,Bi 209_2_1_1,Th 232_2_1_1,U 238_2_1_1,Inscription
14,ISic000003,Catina,NaN,NaN,funerary,chiselled,Latin,"=HYPERLINK(""https://apheleia.classics.ox.ac.uk...",Museo Archeologico Regionale Antonino Salinas,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"=HYPERLINK(""http://sicily.classics.ox.ac.uk/in..."
15,ISic000004,Centuripae,NaN,NaN,building,chiselled,Latin,"=HYPERLINK(""https://apheleia.classics.ox.ac.uk...",Museo Archeologico Regionale Antonino Salinas,NaN,...,0.14,0.34,0.28,0.10088,0.02,5.47,4.54,12.07,0.05,"=HYPERLINK(""http://sicily.classics.ox.ac.uk/in..."
16,ISic000006,Cossura,NaN,NaN,funerary,chiselled,Latin,"=HYPERLINK(""https://apheleia.classics.ox.ac.uk...",Museo Archeologico Regionale Antonino Salinas,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"=HYPERLINK(""http://sicily.classics.ox.ac.uk/in..."
17,ISic000027,Panhormus,NaN,NaN,funerary,chiselled,Latin,"=HYPERLINK(""https://apheleia.classics.ox.ac.uk...",Museo Archeologico Regionale Antonino Salinas,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"=HYPERLINK(""http://sicily.classics.ox.ac.uk/in..."
18,ISic000034,Panhormus,NaN,NaN,funerary,chiselled,Latin,"=HYPERLINK(""https://apheleia.classics.ox.ac.uk...",Museo Archeologico Regionale Antonino Salinas,NaN,...,0.16,0.43,0.34,0.121543,0.01,7.68,4.06,5.21,0.03,"=HYPERLINK(""http://sicily.classics.ox.ac.uk/in..."


In [16]:
get_isic_rows(sd_df).head()

,ISic,Ancient place,Date after,Date before,Type of text,Execution,Language,Link to image,Museum,ref,...,Nb,Pb,Rb,Sr,Th,Ti,Y,Zn,Zr,Inscription
2,ISic000007,Lilybaeum,NaN,NaN,building,chiselled,Latin,"=HYPERLINK(""https://apheleia.classics.ox.ac.uk...",Museo archeologico regionale Lilibeo Marsala -...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"=HYPERLINK(""http://sicily.classics.ox.ac.uk/in..."
3,ISic000009,Panhormus,NaN,NaN,dedication,chiselled,Latin,"=HYPERLINK(""https://apheleia.classics.ox.ac.uk...",Museo Archeologico Regionale Antonino Salinas,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"=HYPERLINK(""http://sicily.classics.ox.ac.uk/in..."
4,ISic000009,Panhormus,NaN,NaN,dedication,chiselled,Latin,"=HYPERLINK(""https://apheleia.classics.ox.ac.uk...",Museo Archeologico Regionale Antonino Salinas,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"=HYPERLINK(""http://sicily.classics.ox.ac.uk/in..."
5,ISic000015,Panhormus,NaN,NaN,honorific,chiselled,Latin,"=HYPERLINK(""https://apheleia.classics.ox.ac.uk...",Museo Archeologico Regionale Antonino Salinas,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"=HYPERLINK(""http://sicily.classics.ox.ac.uk/in..."
6,ISic000017,Panhormus,NaN,NaN,honorific,chiselled,Latin,"=HYPERLINK(""https://apheleia.classics.ox.ac.uk...",Museo Archeologico Regionale Antonino Salinas,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"=HYPERLINK(""http://sicily.classics.ox.ac.uk/in..."


### XRD data

In [17]:
xrd_path = os.path.join(data_path, 'XRD/output/xrd_data_postprocessed_sums.xlsx')
xrd_df = pd.read_excel(xrd_path)
xrd_df.columns = xrd_df.columns.str.strip().str.lower()
xrd_df['other'] = xrd_df['other'].fillna('')

xrd_df.head()

,sample,calcite,quartz,dolomite,pyroxenes,sulphates,clay minerals,plagioclases,alkali_feldspars,fe-oxihydroxides,aragonite,magnesian calcite,olivine,other,total
0,ISic000004,96.58,NaN,NaN,3.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,100.00
1,ISic000009grey,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,100.00
2,ISic000009pink,93.55,NaN,2.55,NaN,NaN,2.21,NaN,NaN,NaN,NaN,NaN,NaN,Qhydrotalcite; Qsiderite,98.31
3,ISic000015,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,100.00
4,ISic000017,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,100.00


In [18]:
def get_xrd_mineral_content(isic, colour, mineral):
    rows = xrd_df[xrd_df['sample'] == f'{isic}{colour}']

    if rows.empty or rows is None:
        rows = xrd_df[xrd_df['sample'].str.startswith(isic)]

    return rows[mineral].mean()

assert get_xrd_mineral_content('ISic000004', 'white', 'calcite') == 96.58
assert get_xrd_mineral_content('ISic000009', 'pink', 'calcite') == 93.55

In [19]:
def get_xrd_other_content(isic, colour):
    rows = xrd_df[xrd_df['sample'] == f'{isic}{colour}']

    if rows.empty or rows is None:
        rows = xrd_df[xrd_df['sample'].str.startswith(isic)]

    other = py_.flatten(rows['other'].str.split('; '))

    return '; '.join([q.replace('Q', '') for q in other])

assert get_xrd_other_content('ISic000131', '') == "anhydrite; hercynite; chiolite; ankerit02"

In [20]:
def get_carbonate_content(x):
    calcite = 0 if np.isnan(x['Calcite']) else x['Calcite']
    mag_calcite = 0 if np.isnan(x['Magnesian calcite']) else x['Magnesian calcite']
    dolomite = 0 if np.isnan(x['Dolomite']) else x['Dolomite']

    return calcite + mag_calcite + dolomite

In [21]:
xrd_minerals = {
    'Calcite': 'calcite',
    'Magnesian calcite': 'magnesian calcite',
    'Dolomite': 'dolomite',
    'Aragonite': 'aragonite',
    'Quartz': 'quartz',
    'Fe-oxihydroxides': 'fe-oxihydroxides',
    'Clay Minerals': 'clay minerals',
    'Alkali_Feldspar': 'alkali_feldspars',
    'Plagioclase': 'plagioclases',
    'Pyroxene': 'pyroxenes',
    'Sulphates': 'sulphates',
    # 'Sulphides': 'pyrite',
    'Olivine': 'olivine'
}

colour_columns = ['visually assessed colour', 'visually assessed colour (fresh cut)']

for i, df in enumerate([mm_df, sd_df]):
    for k, v in xrd_minerals.items():
        df[k] = df.apply(
            lambda x: get_xrd_mineral_content(x['ISic'], x[colour_columns[i]], v),
            axis=1
        )

    df['other'] = df.apply(
        lambda x: get_xrd_other_content(x['ISic'], x[colour_columns[i]]),
        axis=1
    )

    df['XRD carbonate content (%)'] = df.apply(get_carbonate_content, axis=1)

mineral_columns = list(xrd_minerals.keys())
mineral_columns.insert(0, 'ISic')
mineral_columns.insert(1, 'XRD carbonate content (%)')
mineral_columns.append('other')

In [22]:
get_isic_rows(mm_df)[mineral_columns].head()

,ISic,XRD carbonate content (%),Calcite,Magnesian calcite,Dolomite,Aragonite,Quartz,Fe-oxihydroxides,Clay Minerals,Alkali_Feldspar,Plagioclase,Pyroxene,Sulphates,Olivine,other
14,ISic000003,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
15,ISic000004,96.58,96.58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.42,NaN,NaN,
16,ISic000006,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
17,ISic000027,97.13,94.89,NaN,2.24,NaN,NaN,NaN,NaN,NaN,NaN,2.87,NaN,NaN,
18,ISic000034,99.74,98.17,NaN,1.57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,graphite3R


In [23]:
get_isic_rows(sd_df)[mineral_columns].head()

,ISic,XRD carbonate content (%),Calcite,Magnesian calcite,Dolomite,Aragonite,Quartz,Fe-oxihydroxides,Clay Minerals,Alkali_Feldspar,Plagioclase,Pyroxene,Sulphates,Olivine,other
2,ISic000007,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
3,ISic000009,96.1,93.55,NaN,2.55,NaN,NaN,NaN,2.21,NaN,NaN,NaN,NaN,NaN,hydrotalcite; siderite
4,ISic000009,100.0,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
5,ISic000015,100.0,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
6,ISic000017,100.0,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,


### XRF

In [24]:
xrf_path = os.path.join(data_path, 'pXRF/output/pXRF_corrected_values_with_descriptions_mean.csv')
xrf_df = pd.read_csv(xrf_path)
xrf_df.columns = xrf_df.columns.str.strip().str.lower()
xrf_df = xrf_df.drop(columns=["unnamed: 0"])
xrf_df['isic'] = xrf_df['isic'].apply(
    lambda x: x.replace('Isic', 'ISic') if x.startswith('Isic') else x
)
xrf_df['desc'] = xrf_df['desc'].fillna('')

xrf_df.head()

,isic,desc,sio2,tio2,fe2o3,mno,cao,k2o,ti,fe (ppm),...,sr,v,y,zn,zr,au,hg,as,cu,errors
0,0,a = 1,88.950227,0.076902,3.129968,0.032597,0.992413,6.817892,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Invalid source name: 0-1
1,0,b = 2,89.154060,0.086559,3.106822,0.033856,0.946590,6.672112,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Invalid source name: 0-2
2,0,c = 3,88.596589,0.086617,3.268474,0.035490,1.014460,6.998368,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Invalid source name: 0-3
3,10,a = 1,77.013579,0.081111,3.175449,0.034341,14.498359,5.197161,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Invalid source name: 10-1
4,10,b = 2,77.797252,0.088929,3.063678,0.031798,13.866996,5.151347,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Invalid source name: 10-2


In [25]:
def get_xrf_mineral_content(isic, colour, mineral):
    rows = xrf_df[(xrf_df['isic'] == isic) & (xrf_df['desc'] == colour)]

    if rows.empty or rows is None:
        rows = xrf_df[xrf_df['isic'] == isic]

    return rows[mineral.lower()].mean()

assert get_xrf_mineral_content("ISic000003", "white", "SiO2") == 5.381512735095491
assert get_xrf_mineral_content("ISic000004", "grey", "SiO2") == 50.75200775992982

In [26]:
# Duplicate rows in the metamorphic dataset that have multiple colours
# in the XRF data, and update the `visually assessed colour` column
# to include all the colours.
#
# This is a workaround to the fact that the XRF data has multiple
# colours for some inscriptions, and we want to include all of them
# in the final dataset.
xrf_colours_df = xrf_df[['isic', 'desc']]

mm_df['visually assessed colour'] = mm_df.apply(
    lambda x: list(xrf_colours_df[xrf_colours_df['isic'] == x['ISic']]['desc']),
    axis = 1
)

mm_df = mm_df.explode('visually assessed colour').fillna('')

In [27]:
xrf_minerals = [
    "SiO2",
    "TiO2",
    "Fe2O3",
    "MnO",
    "CaO",
    "K2O",
    "Ti",
    "Fe (ppm)",
    "Fe2O3 (t)",
    "Mn",
    "Ca",
    "K",
    "Ba",
    "Co",
    "Cr",
    "Ni",
    "Pb",
    "Rb",
    "Sr",
    "V",
    "Y",
    "Zn",
    "Zr",
    "Au",
    "Hg",
    "As",
    "Cu",
]

colour_columns = ['visually assessed colour', 'visually assessed colour (fresh cut)']

for i, df in enumerate([mm_df, sd_df]):
    for column in xrf_minerals:
        df[column] = df.apply(
            lambda x: get_xrf_mineral_content(x['ISic'], x[colour_columns[i]], column),
            axis=1
        )

xrf_minerals.insert(0, 'ISic')

In [28]:
get_isic_rows(mm_df)[xrf_minerals].head()

,ISic,SiO2,TiO2,Fe2O3,MnO,CaO,K2O,Ti,Fe (ppm),Fe2O3 (t),...,Rb,Sr,V,Y,Zn,Zr,Au,Hg,As,Cu
14,ISic000003,5.381513,0.066648,0.154162,0.008967,94.361326,0.027384,0.074695,783.066667,0.127868,...,7.533124,72.326464,17.079667,9.347258,22.705378,11.139292,7.340277,7.946968,7.441732,65.300000
15,ISic000004,50.752008,0.018070,0.055579,0.006802,49.135587,0.031954,0.026013,325.466667,0.053110,...,6.610856,507.199786,11.927333,10.130892,22.681163,5.118830,8.588278,10.045708,6.374699,63.410000
15,ISic000004,50.950587,0.033522,0.063789,0.006360,48.916380,0.029362,0.030025,413.593333,0.067510,...,6.469859,495.280461,26.163333,9.619576,38.213599,5.735918,9.132243,7.826315,10.579094,119.706667
16,ISic000006,8.444394,0.040552,0.148669,0.011576,91.302003,0.052806,0.027463,681.000000,0.111185,...,6.687145,143.907429,0.763433,8.459550,24.246514,11.691642,8.783310,6.079936,7.387144,69.530000
17,ISic000027,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
get_isic_rows(sd_df)[xrf_minerals].head()

,ISic,SiO2,TiO2,Fe2O3,MnO,CaO,K2O,Ti,Fe (ppm),Fe2O3 (t),...,Rb,Sr,V,Y,Zn,Zr,Au,Hg,As,Cu
2,ISic000007,NaN,NaN,NaN,NaN,NaN,NaN,0.011771,1464.066667,0.239307,...,9.692007,481.150643,19.427333,11.71914,33.917099,13.239304,8.66917,8.764458,20.797552,75.17
3,ISic000009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ISic000009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ISic000015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ISic000017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Save aggregated data

In [30]:
try:
    mm_df.loc[0] = mm_row_0
    mm_df.to_csv(os.path.join(data_path, 'Metadata/Metamorphic_with_data.csv'))
    mm_df.to_excel(os.path.join(data_path, 'Metadata/Metamorphic_with_data.xlsx'))

    sd_df.to_csv(os.path.join(data_path, 'Metadata/Sedimentary_with_data.csv'))
    sd_df.to_excel(os.path.join(data_path, 'Metadata/Sedimentary_with_data.xlsx'))
except FutureWarning:
    pass

/var/folders/y3/xd80lxdn6bv9ghx4stmsk7900000gp/T/ipykernel_85855/3708901512.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'sum function of Calcite, Magnesian calcite and Dolomite' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mm_df.loc[0] = mm_row_0
/var/folders/y3/xd80lxdn6bv9ghx4stmsk7900000gp/T/ipykernel_85855/3708901512.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'XRF TOOL (header in pXRF_calculated_fractions_mean.xlsx)' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mm_df.loc[0] = mm_row_0
/var/folders/y3/xd80lxdn6bv9ghx4stmsk7900000gp/T/ipykernel_85855/3708901512.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'XRF TOOL (header in pX